In [2]:
import pandas as pd
df=pd.read_csv('retail_sales.csv')
print('Data Loaded Successfully')
print('Rows:',df.shape[0])
print('Columns:',df.shape[1])


Data Loaded Successfully
Rows: 9994
Columns: 21


In [4]:
# Check Order_Date
print(df['Order Date'].dtype)
# Convert Order_Date to datetime format
df['Order Date']=pd.to_datetime(df['Order Date'])
print(df['Order Date'].dtype)

str
datetime64[us]


In [5]:
# Create Year from Order_Date
df['Year']=df['Order Date'].dt.year
print(df[['Order Date','Year']].head())

  Order Date  Year
0 2016-11-08  2016
1 2016-11-08  2016
2 2016-06-12  2016
3 2015-10-11  2015
4 2015-10-11  2015


In [6]:
# Create Month from Order_Date
df['Month']=df['Order Date'].dt.month
print(df[['Order Date','Month']].head())

  Order Date  Month
0 2016-11-08     11
1 2016-11-08     11
2 2016-06-12      6
3 2015-10-11     10
4 2015-10-11     10


In [7]:
# Create Month Name from order date
df['Month_Name']=df['Order Date'].dt.month_name()
print(df[['Order Date','Month','Month_Name']].head())


  Order Date  Month Month_Name
0 2016-11-08     11   November
1 2016-11-08     11   November
2 2016-06-12      6       June
3 2015-10-11     10    October
4 2015-10-11     10    October


In [8]:
# Create Quarter from Order Date
df['Quarter']=df['Order Date'].dt.quarter
print(df[['Order Date','Month','Quarter']].head())

  Order Date  Month  Quarter
0 2016-11-08     11        4
1 2016-11-08     11        4
2 2016-06-12      6        2
3 2015-10-11     10        4
4 2015-10-11     10        4


In [ ]:
#check column names
print(df.columns.tolist())

['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit', 'Year', 'Month', 'Month_Name', 'Quarter']


In [10]:
#calculate profit margin
df['Profit_Margin']=(df['Profit']/df['Sales'])*100
print(df[['Sales','Profit','Profit_Margin']].head())

      Sales    Profit  Profit_Margin
0  261.9600   41.9136          16.00
1  731.9400  219.5820          30.00
2   14.6200    6.8714          47.00
3  957.5775 -383.0310         -40.00
4   22.3680    2.5164          11.25


In [15]:
# Create Order Value using the available Sales field
df["Order_Value"] = df["Sales"]

print(df[["Order ID", "Sales", "Order_Value"]].head())


         Order ID     Sales  Order_Value
0  CA-2016-152156  261.9600     261.9600
1  CA-2016-152156  731.9400     731.9400
2  CA-2016-138688   14.6200      14.6200
3  US-2015-108966  957.5775     957.5775
4  US-2015-108966   22.3680      22.3680


In [ ]:
# Customer Segmentation 
# Calculate total spending for each customer
customer_spending=df.groupby('Customer ID')['Sales'].sum()
print(customer_spending.head())

Customer ID
AA-10315    5563.560
AA-10375    1056.390
AA-10480    1790.512
AA-10645    5086.935
AB-10015     886.156
Name: Sales, dtype: float64


In [17]:
# Find spending thresholds for customer segmentation
low_limit=customer_spending.quantile(0.25)
high_limit=customer_spending.quantile(0.75)

print('Low value limit:',low_limit)
print('High value limit:',high_limit)

Low value limit: 1146.05
High value limit: 3785.276


In [19]:
# Create customer segments based on total spending
def Customer_Segment(customer_id):
    spending=customer_spending[customer_id]
    if spending<=low_limit:
        return 'Low value'
    elif spending<=high_limit:
        return 'Medium value'
    else:
        return 'High value'
df['Customer_Segment']=df['Customer ID'].apply(Customer_Segment)
print(df[['Customer ID','Sales','Customer_Segment']].head())

  Customer ID     Sales Customer_Segment
0    CG-12520  261.9600     Medium value
1    CG-12520  731.9400     Medium value
2    DV-13045   14.6200        Low value
3    SO-20335  957.5775     Medium value
4    SO-20335   22.3680     Medium value


In [ ]:
# Additional Analytical
# 1.Sales Category
# Find sales thresholds
sales_low=df['Sales'].quantile(0.33)#We use 0.33 and 0.66 because we want to divide the Sales
sales_high=df['Sales'].quantile(0.66)   #values into approximately three group
print('Low Sales Limit:',sales_low)
print('High Sales Limit:',sales_high)

Low Sales Limit: 24.870700000000003
High Sales Limit: 122.9514


In [22]:
# Create Sales_Category
df['Sales_Category']=df['Sales'].apply(
    lambda x:'Low Sales' if x<=sales_low
    else 'Medium Sales' if x<=sales_high
    else 'High Sales'
)
print(df[['Sales','Sales_Category']].head())

      Sales Sales_Category
0  261.9600     High Sales
1  731.9400     High Sales
2   14.6200      Low Sales
3  957.5775     High Sales
4   22.3680      Low Sales


In [23]:
# Profit Category
df['Profit_Category']=df['Profit'].apply(
    lambda x:'High Profit' if x>0 else 'Low Profit'
)
print(df[['Profit','Profit_Category']].head())

     Profit Profit_Category
0   41.9136     High Profit
1  219.5820     High Profit
2    6.8714     High Profit
3 -383.0310      Low Profit
4    2.5164     High Profit


In [24]:
# Customer Type
df['Customer_Type']=df['Customer ID'].apply(
    lambda x:'Existing Customer' if x in customer_spending.index
    else 'New Customer'
)
print(df[['Customer ID','Customer_Type']].head())

  Customer ID      Customer_Type
0    CG-12520  Existing Customer
1    CG-12520  Existing Customer
2    DV-13045  Existing Customer
3    SO-20335  Existing Customer
4    SO-20335  Existing Customer


In [25]:
# Product Performance
df['Product_Performance']=df['Profit'].apply(
    lambda x:'Good Performance' if x>0 else 'Poor Performance'
)
print(df[['Product Name','Profit','Product_Performance']].head())

                                        Product Name    Profit  \
0                  Bush Somerset Collection Bookcase   41.9136   
1  Hon Deluxe Fabric Upholstered Stacking Chairs,...  219.5820   
2  Self-Adhesive Address Labels for Typewriters b...    6.8714   
3      Bretford CR4500 Series Slim Rectangular Table -383.0310   
4                     Eldon Fold 'N Roll Cart System    2.5164   

  Product_Performance  
0    Good Performance  
1    Good Performance  
2    Good Performance  
3    Poor Performance  
4    Good Performance  


In [28]:

# Order_Size
df['Order_Size']=df['Quantity'].apply(
    lambda x:'Small Order' if x<=2
    else 'Medium Order' if x<=5
    else 'Larger Order'
)
print(df[['Quantity','Order_Size']].head())

   Quantity    Order_Size
0         2   Small Order
1         3  Medium Order
2         2   Small Order
3         5  Medium Order
4         2   Small Order


In [29]:
print(df.columns.tolist())

['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit', 'Year', 'Month', 'Month_Name', 'Quarter', 'Profit_Margin', 'Order_value', 'Order_Value', 'Customer_segment', 'Customer_Segment', 'Sales_Category', 'Profit_Category', 'Customer_Type', 'Product_Performance', 'Order_Size']


In [30]:
df = df.drop(columns=["Order_value", "Customer_segment"])

print(df.columns.tolist())

['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit', 'Year', 'Month', 'Month_Name', 'Quarter', 'Profit_Margin', 'Order_Value', 'Customer_Segment', 'Sales_Category', 'Profit_Category', 'Customer_Type', 'Product_Performance', 'Order_Size']


In [31]:
print(df.columns.tolist())

['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit', 'Year', 'Month', 'Month_Name', 'Quarter', 'Profit_Margin', 'Order_Value', 'Customer_Segment', 'Sales_Category', 'Profit_Category', 'Customer_Type', 'Product_Performance', 'Order_Size']


In [32]:
# check missing values
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Missing Values:", df.isnull().sum().sum())

Rows: 9994
Columns: 33
Missing Values: 0


In [33]:
df.to_csv("transformed_retail_sales.csv", index=False)

print("Transformed dataset saved successfully!")

Transformed dataset saved successfully!


In [34]:
check_df = pd.read_csv("transformed_retail_sales.csv")

print("File loaded successfully!")
print("Rows:", check_df.shape[0])
print("Columns:", check_df.shape[1])

File loaded successfully!
Rows: 9994
Columns: 33


In [ ]:
# Final Summary Table
summary = pd.DataFrame({
    "Transformation": [
        "Date Features",
        "Profit Margin",
        "Order Value",
        "Customer Segmentation",
        "Sales Category",
        "Profit Category",
        "Customer Type",
        "Product Performance",
        "Order Size"
    ],
    "New Columns": [
        "Year, Month, Month_Name, Quarter",
        "Profit_Margin",
        "Order_Value",
        "Customer_Segment",
        "Sales_Category",
        "Profit_Category",
        "Customer_Type",
        "Product_Performance",
        "Order_Size"
    ]
})

display(summary)

,Transformation,New Columns
0,Date Features,"Year, Month, Month_Name, Quarter"
1,Profit Margin,Profit_Margin
2,Order Value,Order_Value
3,Customer Segmentation,Customer_Segment
4,Sales Category,Sales_Category
5,Profit Category,Profit_Category
6,Customer Type,Customer_Type
7,Product Performance,Product_Performance
8,Order Size,Order_Size
